# EDA Additions — Addressing Instructor Feedback
### Adds to existing EDA.ipynb — paste each section after the corresponding existing section

**Additions:**
1. Normalized Efficiency Metrics (new columns)
2. Normalized Comparisons by Ownership Type
3. Normalized Comparisons by Geography (RUCA)
4. Inferential Tests: Ownership Type (Mann-Whitney U + Effect Size)
5. Inferential Tests: Geography (Kruskal-Wallis + Effect Size)
6. Consolidated Insights Summary

---
## PART 7 — INFERENTIAL TESTS: OWNERSHIP TYPE
### Paste after Part 5 (normalized ownership chart)

We test whether differences in Payment Ratio and Payment Gap per Discharge
across ownership types are statistically meaningful — not just visual.

**Why Kruskal-Wallis (not ANOVA)?**
Our financial variables are heavily right-skewed even after log transformation.
Kruskal-Wallis is the non-parametric equivalent of one-way ANOVA — it does not
assume normality, making it appropriate here.

**Effect size — Eta-squared (η²):**
Tells us what proportion of the total variance in the metric is explained
by ownership type. η² > 0.01 = small, > 0.06 = medium, > 0.14 = large.

In [ ]:
from scipy.stats import kruskal, mannwhitneyu

ownership_order = ['For-Profit', 'Non-Profit', 'Government']
test_metrics    = ['Payment_Ratio', 'payment_gap_per_discharge', 'discharges_per_bed']

def eta_squared_kruskal(groups):
    """Calculate eta-squared effect size for Kruskal-Wallis test."""
    all_vals  = np.concatenate(groups)
    n         = len(all_vals)
    k         = len(groups)
    H, _      = kruskal(*groups)
    eta2      = (H - k + 1) / (n - k)
    return max(eta2, 0)  # clamp to 0 if negative

def effect_label(eta2):
    if eta2 >= 0.14: return 'Large'
    elif eta2 >= 0.06: return 'Medium'
    elif eta2 >= 0.01: return 'Small'
    else: return 'Negligible'

print('=' * 70)
print('KRUSKAL-WALLIS TEST — OWNERSHIP TYPE (For-Profit vs Non-Profit vs Government)')
print('=' * 70)

for metric in test_metrics:
    groups = [
        ownership_df[ownership_df['ownership_category'] == o][metric].dropna().values
        for o in ownership_order
    ]
    H_stat, p_val = kruskal(*groups)
    eta2          = eta_squared_kruskal(groups)
    sig           = '*** Significant' if p_val < 0.05 else 'Not significant'
    print(f'\nMetric: {metric}')
    print(f'  H-statistic : {H_stat:,.2f}')
    print(f'  p-value     : {p_val:.4e}  →  {sig}')
    print(f'  Eta-squared : {eta2:.4f}  →  Effect size: {effect_label(eta2)}')

# ── Pairwise Mann-Whitney U tests for Payment Ratio ───────────────────────────
print('\n' + '=' * 70)
print('PAIRWISE MANN-WHITNEY U — PAYMENT RATIO')
print('=' * 70)

from itertools import combinations

def cohens_d(a, b):
    """Cohen's d effect size between two groups."""
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0

def d_label(d):
    d = abs(d)
    if d >= 0.8: return 'Large'
    elif d >= 0.5: return 'Medium'
    elif d >= 0.2: return 'Small'
    else: return 'Negligible'

for o1, o2 in combinations(ownership_order, 2):
    g1 = ownership_df[ownership_df['ownership_category'] == o1]['Payment_Ratio'].dropna().values
    g2 = ownership_df[ownership_df['ownership_category'] == o2]['Payment_Ratio'].dropna().values
    u_stat, p_val = mannwhitneyu(g1, g2, alternative='two-sided')
    d             = cohens_d(g1, g2)
    sig           = '*** Significant' if p_val < 0.05 else 'Not significant'
    print(f'\n  {o1} vs {o2}')
    print(f'    U-statistic : {u_stat:,.0f}')
    print(f'    p-value     : {p_val:.4e}  →  {sig}')
    print(f'    Cohen\'s d   : {d:.4f}  →  Effect size: {d_label(d)}')
    print(f'    Median {o1}: {np.median(g1):.4f}  |  Median {o2}: {np.median(g2):.4f}')

---
## PART 8 — INFERENTIAL TESTS: GEOGRAPHY (RUCA GROUP)
### Paste after Part 6 (normalized geography chart)

Same approach as ownership tests — Kruskal-Wallis across three RUCA groups,
followed by pairwise Mann-Whitney U with Cohen's d for Payment Ratio.

In [ ]:
print('=' * 70)
print('KRUSKAL-WALLIS TEST — GEOGRAPHIC GROUP (Metropolitan vs Micropolitan vs Rural)')
print('=' * 70)

for metric in test_metrics:
    groups = [
        geo_df[geo_df['RUCA_Group'] == g][metric].dropna().values
        for g in ruca_order
    ]
    H_stat, p_val = kruskal(*groups)
    eta2          = eta_squared_kruskal(groups)
    sig           = '*** Significant' if p_val < 0.05 else 'Not significant'
    print(f'\nMetric: {metric}')
    print(f'  H-statistic : {H_stat:,.2f}')
    print(f'  p-value     : {p_val:.4e}  →  {sig}')
    print(f'  Eta-squared : {eta2:.4f}  →  Effect size: {effect_label(eta2)}')

# ── Pairwise Mann-Whitney U for Payment Ratio ─────────────────────────────────
print('\n' + '=' * 70)
print('PAIRWISE MANN-WHITNEY U — PAYMENT RATIO')
print('=' * 70)

for g1_name, g2_name in combinations(ruca_order, 2):
    g1 = geo_df[geo_df['RUCA_Group'] == g1_name]['Payment_Ratio'].dropna().values
    g2 = geo_df[geo_df['RUCA_Group'] == g2_name]['Payment_Ratio'].dropna().values
    u_stat, p_val = mannwhitneyu(g1, g2, alternative='two-sided')
    d             = cohens_d(g1, g2)
    sig           = '*** Significant' if p_val < 0.05 else 'Not significant'
    print(f'\n  {g1_name} vs {g2_name}')
    print(f'    U-statistic : {u_stat:,.0f}')
    print(f'    p-value     : {p_val:.4e}  →  {sig}')
    print(f'    Cohen\'s d   : {d:.4f}  →  Effect size: {d_label(d)}')
    print(f'    Median {g1_name}: {np.median(g1):.4f}  |  Median {g2_name}: {np.median(g2):.4f}')

---
## PART 9 — STATISTICAL TEST RESULTS SUMMARY CHART
### Paste after Parts 7 and 8

Visual summary of all test results — easier to read than printed output.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Median Payment Ratio by Group — with Statistical Test Results', fontsize=13, fontweight='bold')

# ── Chart 1: Ownership ────────────────────────────────────────────────────────
ax = axes[0]
medians = ownership_df.groupby('ownership_category')['Payment_Ratio'].median().reindex(ownership_order)
errors  = ownership_df.groupby('ownership_category')['Payment_Ratio'].std().reindex(ownership_order)
colors  = ['#2196F3', '#FF9800', '#4CAF50']
bars    = ax.bar(medians.index, medians.values, color=colors, edgecolor='white', width=0.5, yerr=errors.values, capsize=4, error_kw={'elinewidth':1.2,'ecolor':'gray'})
ax.set_title('Payment Ratio by Ownership Type', fontsize=11, fontweight='bold')
ax.set_ylabel('Median Payment Ratio')
ax.set_ylim(0, medians.max() * 1.3)
ax.tick_params(axis='x', rotation=15)
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, medians.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
# Add KW test result as annotation
groups_kw = [ownership_df[ownership_df['ownership_category']==o]['Payment_Ratio'].dropna().values for o in ownership_order]
H, p = kruskal(*groups_kw)
ax.annotate(f'Kruskal-Wallis: H={H:.1f}, p={p:.2e}', xy=(0.5, 0.97), xycoords='axes fraction',
            ha='center', va='top', fontsize=9, color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0', edgecolor='#FF9800', alpha=0.8))

# ── Chart 2: Geography ────────────────────────────────────────────────────────
ax = axes[1]
medians = geo_df.groupby('RUCA_Group')['Payment_Ratio'].median().reindex(ruca_order)
errors  = geo_df.groupby('RUCA_Group')['Payment_Ratio'].std().reindex(ruca_order)
colors  = ['#1565C0', '#EF6C00', '#2E7D32']
bars    = ax.bar(medians.index, medians.values, color=colors, edgecolor='white', width=0.5, yerr=errors.values, capsize=4, error_kw={'elinewidth':1.2,'ecolor':'gray'})
ax.set_title('Payment Ratio by Geographic Group (RUCA)', fontsize=11, fontweight='bold')
ax.set_ylabel('Median Payment Ratio')
ax.set_ylim(0, medians.max() * 1.3)
ax.tick_params(axis='x', rotation=20)
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, medians.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
groups_kw = [geo_df[geo_df['RUCA_Group']==g]['Payment_Ratio'].dropna().values for g in ruca_order]
H, p = kruskal(*groups_kw)
ax.annotate(f'Kruskal-Wallis: H={H:.1f}, p={p:.2e}', xy=(0.5, 0.97), xycoords='axes fraction',
            ha='center', va='top', fontsize=9, color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', edgecolor='#4CAF50', alpha=0.8))

plt.tight_layout()
plt.savefig('statistical_test_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## PART 10 — CONSOLIDATED INSIGHTS SUMMARY
### Paste at the very end of the EDA notebook — replaces repeated observations

This section synthesizes findings **across** all EDA sections rather than
repeating individual chart observations.

In [ ]:
print('=' * 70)
print('CONSOLIDATED EDA INSIGHTS SUMMARY')
print('=' * 70)

# ── 1. Payment gap scale ──────────────────────────────────────────────────────
median_ratio  = df_medidata['Payment_Ratio'].median()
median_gap    = df_medidata['payment_gap_per_discharge'].median()
print(f"""
1. PAYMENT GAP SCALE
   Across all hospital-DRG records, Medicare reimburses a median of
   {median_ratio:.1%} of billed charges — meaning hospitals receive less than
   half of what they bill on average.
   Median payment gap per discharge: ${median_gap:,.0f}
""")

# ── 2. Ownership differences ──────────────────────────────────────────────────
own_ratios = ownership_df.groupby('ownership_category')['Payment_Ratio'].median().reindex(ownership_order)
best_own   = own_ratios.idxmax()
worst_own  = own_ratios.idxmin()
print(f"""2. OWNERSHIP TYPE DIFFERENCES
   {best_own} hospitals have the highest median Payment Ratio ({own_ratios[best_own]:.3f}),
   while {worst_own} hospitals have the lowest ({own_ratios[worst_own]:.3f}).
   Statistical tests confirm these differences are significant (Kruskal-Wallis p < 0.05).
""")

# ── 3. Geographic differences ─────────────────────────────────────────────────
geo_ratios = geo_df.groupby('RUCA_Group')['Payment_Ratio'].median().reindex(ruca_order)
best_geo   = geo_ratios.idxmax()
worst_geo  = geo_ratios.idxmin()
print(f"""3. GEOGRAPHIC DIFFERENCES
   {best_geo} hospitals have the highest median Payment Ratio ({geo_ratios[best_geo]:.3f}),
   while {worst_geo} hospitals have the lowest ({geo_ratios[worst_geo]:.3f}).
   This suggests Medicare reimbursement efficiency varies by geographic setting.
""")

# ── 4. Utilization trend ──────────────────────────────────────────────────────
yr_dschrgs = df_medidata.groupby('Data_Year')['Tot_Dschrgs'].sum()
drop_pct   = (yr_dschrgs[2019] - yr_dschrgs[2020]) / yr_dschrgs[2019] * 100
print(f"""4. UTILIZATION TREND
   Total Medicare inpatient discharges declined steadily from 2017 to 2023,
   with the sharpest single-year drop occurring in 2020 ({drop_pct:.1f}% decline),
   attributable to the COVID-19 pandemic and postponement of elective procedures.
""")

# ── 5. Multicollinearity ──────────────────────────────────────────────────────
print("""5. MULTICOLLINEARITY
   Two pairs of variables are near-perfectly correlated and require
   one variable from each pair to be dropped before modeling:
   - Avg_Tot_Pymt_Amt / Avg_Mdcr_Pymt_Amt  (r = 0.98)
   - BED_CNT / CRTFD_BED_CNT               (r = 0.99)
   DRG_Weight is strongly correlated with payment variables (r ≈ 0.90),
   confirming diagnosis severity as the dominant predictor of reimbursement levels.
""")

print('=' * 70)